## TOP SET-BASED GENE RESULTS 

In [ ]:
list.files(path="merged_saige_gene_step02")

In [ ]:
library(tidyverse)
library(ggplot2)
library(data.table)
library(forcats)
library(flextable)
library(officer)

In [ ]:
EUR_aml_phenotype_merged_chr <- fread("merged_saige_gene_step02/EUR_aml_phenotype_high_confidence_merged_chr")

In [ ]:
EUR_aml_phenotype_merged_chr <- 
EUR_aml_phenotype_merged_chr %>% 
filter(Group != "Cauchy")  %>% 
mutate(max_MAF = as.character(max_MAF)) %>% 
mutate(negative_log_p_values = -log10(Pvalue)) %>% 
mutate(Chromosome = gsub("chr", "", Chromosome)) %>% 
mutate(Chromosome = as.numeric(Chromosome)) %>% 
mutate(Region_factor = as.factor(Region)) %>% 
mutate(Region_factor = fct_reorder(Region, Chromosome))


In [ ]:
value <- 0.05/nrow(EUR_aml_phenotype_merged_chr)
ntests <- nrow(EUR_aml_phenotype_merged_chr)
EUR_aml_phenotype_genes <- 
EUR_aml_phenotype_merged_chr %>% 
    mutate(Bonferroni_Pvalue = (Pvalue * ntests)) %>% 
    filter(Pvalue < value) %>% 
    select(Chromosome, Region, Group, Pvalue, Pvalue_Burden, Pvalue_SKAT, Bonferroni_Pvalue, BETA_Burden, SE_Burden) 
colnames(EUR_aml_phenotype_genes)[4] <- "SKATO P-values"
colnames(EUR_aml_phenotype_genes)[5] <- "Burden P-values"
colnames(EUR_aml_phenotype_genes)[6] <- "SKAT P-values"
colnames(EUR_aml_phenotype_genes)[7] <- "Bonferroni SKATO P-values"
colnames(EUR_aml_phenotype_genes)[8] <- "Burden Beta"
colnames(EUR_aml_phenotype_genes)[9] <- "Standard Error of Burden Beta"
ft <- flextable(EUR_aml_phenotype_genes)
ppt <- read_pptx() %>%
  add_slide(layout = "Title and Content", master = "Office Theme") %>%
  ph_with(value = ft, location = ph_location_type(type = "body"))
print(ppt, target = "EUR_aml_phenotype_genes.pptx")
